<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/dentistry/lecture_6/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_%E2%84%966_%D0%9E%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B5%D0%B9_%D0%BA%D0%B0%D0%BA_%D1%8D%D1%82%D0%BE_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0%D0%B5%D1%82_(%D0%B4%D0%BB%D1%8F_%D0%BA%D0%BB%D0%B8%D0%BD%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B9_%D1%81%D1%82%D0%BE%D0%BC%D0%B0%D1%82%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D0%B8_%D0%B8_%D1%81%D0%BC%D0%B5%D0%B6%D0%BD%D1%8B%D1%85_%D1%81%D0%BF%D0%B5%D1%86%D0%B8%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D0%B5%D0%B9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция №6: Обучение моделей: как это работает (для клинической стоматологии и смежных специальностей)

---

## 1. ВВЕДЕНИЕ

### 1.1. От теории к практике: постановка проблемы

В Лекции 5 мы рассмотрели архитектуру современных NLP-моделей — трансформеров, BERT, GPT — и обсудили концепцию предобучения (pre-training) и дообучения (fine-tuning). Мы узнали, что эти модели изначально обучаются на огромных корпусах текстов (миллиарды слов), а затем адаптируются к конкретным задачам с помощью дообучения на размеченных датасетах.

Однако остался открытым вопрос: **что на самом деле происходит в процессе дообучения?** Когда мы запускаем `trainer.train()` в среде Hugging Face, какие процессы запускаются «под капотом»? Как модель «понимает», что она ошиблась, и каким образом она корректирует свои внутренние параметры?

**Цель данной лекции** — раскрыть механизмы обучения нейросетей на интуитивном уровне, без использования математических формул, но с сохранением научной строгости. Мы рассмотрим:

1. Сущность процесса обучения как настройки параметров модели.
2. Роль функции потерь (loss function) в оценке ошибок модели.
3. Алгоритм градиентного спуска (gradient descent) как метод коррекции параметров.
4. Различие между параметрами (обучаемыми величинами) и гиперпараметрами (настройками, задаваемыми исследователем).
5. Принципы разделения данных на обучающую, валидационную и тестовую выборки.
6. Интерпретацию графиков обучения (кривые loss и accuracy).
7. Систему метрик оценки качества классификации (accuracy, precision, recall, F1) и их применение в клинической стоматологии.

**Ключевая мысль:** понимание процесса обучения — это **ключ к критической оценке** любой ИИ-системы. Если вы понимаете, как модель учится, вы понимаете, **когда ей можно доверять, а когда нет**. В клинической медицине, где цена ошибки особенно высока, это знание становится **этической необходимостью** [7].

---

### 1.2. Актуальность для стоматологии и смежных областей

В последние годы наблюдается активное внедрение методов машинного обучения в стоматологию и челюстно-лицевую хирургию. Согласно обзору [5], количество публикаций по применению NLP в стоматологии выросло в 4 раза за период 2020–2025 годов. Модели используются для:

- Автоматического анализа жалоб пациентов в онлайн-чатах клиник (выявление острой боли, отёка, неотложных состояний).
- Классификации рентгенологических заключений (например, выявление признаков периодонтита, кист, новообразований по текстовым описаниям).
- Оценки риска осложнений после удаления зубов, имплантации, эндодонтического лечения.
- Анализа отзывов пациентов для улучшения качества обслуживания.
- Поддержки принятия решений при дифференциальной диагностике (пульпит vs периодонтит).

Однако, как отмечается в работе [7], многие клиницисты не имеют достаточного понимания принципов работы этих моделей, что создаёт риски необоснованного доверия или, наоборот, неоправданного скепсиса. Настоящая лекция призвана заполнить этот пробел.

---

### 1.3. Структура и методология

Лекция построена по принципу «от общего к частному». Каждый новый концепт вводится через аналогию из клинической практики (например, супервизия ординатора), а затем раскрывается на уровне, достаточном для понимания процессов, происходящих при дообучении моделей. В конце каждого раздела даётся отсылка к Практике 6, где студенты могут наблюдать описанные процессы в действии.

```mermaid
flowchart LR
    A[Что такое<br>обучение модели] --> B[Функция потерь<br>и градиентный спуск]
    B --> C[Параметры vs<br>гиперпараметры]
    C --> D[Разделение<br>данных]
    D --> E[Графики<br>обучения]
    E --> F[Метрики<br>для врача]
    
    style A fill:#e1f5fe,stroke:#01579b
    style B fill:#fff9c4,stroke:#f57f17
    style C fill:#e8f5e9,stroke:#2e7d32
    style D fill:#f3e5f5,stroke:#4a148c
    style E fill:#ffccbc,stroke:#bf360c
    style F fill:#c8e6c9,stroke:#2e7d32
```

---

### 1.4. Связь с предыдущими лекциями и практикой

| Предыдущий материал | Связь с текущей темой |
|---------------------|----------------------|
| **Лекция 5.** Архитектура трансформеров, предобучение и дообучение | Сегодня мы раскрываем **механизмы** дообучения — что происходит внутри модели |
| **Практика 6.** Запуск дообучения, построение графиков | Сегодня мы объясняем **почему** графики выглядят так, а не иначе |
| **Лекция 4.** Визуализация данных | Сегодня мы учимся **интерпретировать** графики обучения |
| **Лекция 3.** Очистка текста и векторизация | Данные для обучения должны быть подготовлены (токенизация, лемматизация) |

---

### 1.5. Этический контекст

В соответствии с принципами ответственного применения ИИ в медицине [7], важно подчеркнуть: модели машинного обучения не являются заменой клиническому мышлению. Они представляют собой инструменты, которые могут **поддерживать** принятие решений, но не **заменять** его. Понимание механизмов обучения позволяет клиницисту:

- Критически оценивать заявленные показатели точности.
- Выявлять потенциальные источники ошибок (смещение выборки, переобучение).
- Принимать обоснованные решения о целесообразности использования модели.
- Объяснять пациентам принципы работы ИИ-систем.

---

## 2. ЧТО ЗНАЧИТ «ОБУЧИТЬ МОДЕЛЬ»

### 2.1. Модель как параметрическая функция

В современном машинном обучении модель представляет собой **параметрическую функцию** — математическое выражение, которое преобразует входные данные (например, текст жалобы) в выходные данные (например, класс заболевания). Параметры функции — это числа, которые определяют характер этого преобразования [1], [2].

**Техническое уточнение:** в случае нейросетей параметрами являются веса связей между нейронами (synaptic weights) и смещения (biases). В модели ruBERT, используемой в Практике 6, таких параметров **178 миллионов** — каждый из них представляет собой число с плавающей точкой.

**Интуитивная аналогия (для понимания):**

> Представьте, что модель — это сложный музыкальный синтезатор с миллионами регуляторов. Каждый регулятор соответствует одному параметру. Изначально регуляторы находятся в случайных положениях — синтезатор издаёт хаотичный шум. В процессе обучения регуляторы постепенно настраиваются так, чтобы синтезатор начинал воспроизводить определённую мелодию.

**Определение обучения:** обучение модели — это процесс последовательной настройки параметров с целью минимизации ошибки на обучающих данных [3].

---

### 2.2. Процесс обучения как итеративная коррекция

**Интуитивная аналогия из клинической практики (расширенная версия):**

> Представьте, что вы — заведующий отделением, и к вам пришёл ординатор. У него есть теоретические знания (аналог предобученной модели), но нет клинического опыта. Вы даёте ему 1000 клинических случаев (обучающий датасет). Для каждого случая он предлагает диагностическую гипотезу, а вы сообщаете правильный диагноз.
>
> Каждый раз, когда он ошибается, он **корректирует свой подход** — не в общем смысле, а в конкретных аспектах: учится замечать определённые паттерны (например, что боль при накусывании и отёк десны характерны для периодонтита, а не для кариеса), различать похожие состояния, обращать внимание на ключевые симптомы. Эти корректировки постепенно накапливаются, и после 1000 случаев его точность значительно возрастает.
>
> Важно понимать: ординатор не «запоминает» случаи (это было бы переобучением). Он извлекает **общие закономерности**, которые позволяют ему работать с новыми, незнакомыми случаями.

**Как это происходит в нейросети:**

1. На вход подаётся пример (текст + правильный диагноз/класс).
2. Модель выполняет прямое распространение (forward pass) — данные проходят через все слои, и на выходе формируется предсказание (вероятности для каждого класса).
3. Вычисляется ошибка (loss) — сравнение предсказания с правильным ответом с помощью функции потерь.
4. Выполняется обратное распространение (backward pass) — вычисляется градиент, показывающий, как нужно изменить каждый параметр, чтобы уменьшить ошибку.
5. Параметры обновляются — делается шаг в направлении, указанном градиентом.
6. Процесс повторяется со следующим примером.

**Связь с Практикой 6:** в нашей практике мы использовали `Trainer` из библиотеки Hugging Face, который автоматизирует все эти шаги. Когда мы запускали `trainer.train()`, выполнялись тысячи итераций этого цикла.

---

### 2.3. Функция потерь (Loss Function): количественная оценка ошибки

**Определение:** функция потерь (loss function) — это математическая функция, которая вычисляет **числовое значение**, отражающее расхождение между предсказанием модели и правильным ответом. Чем больше расхождение, тем выше значение loss [1], [2].

**Техническое уточнение:** в задачах классификации наиболее широко используется **кросс-энтропийная функция потерь** (cross-entropy loss). Она измеряет разницу между распределением вероятностей, предсказанных моделью, и распределением, где правильному классу соответствует вероятность 1. Согласно [3], кросс-энтропия является стандартом для задач классификации благодаря своим хорошим градиентным свойствам.

**Интуитивная аналогия:**

> Представьте, что вы оцениваете работу ординатора на клиническом разборе. Вы не просто говорите «правильно» или «неправильно» — вы ставите количественную оценку:
> - Точно поставил диагноз → 0 баллов ошибки (loss ≈ 0).
> - Ошибся, но близко (например, спутал пульпит с периодонтитом) → 1 балл ошибки (loss = 1).
> - Полностью не прав (например, при явном периодонтите сказал «кариес») → 10 баллов ошибки (loss = 10).
>
> Эта количественная оценка позволяет ординатору понять **степень** своей ошибки и соответственно скорректировать подход.

**Пример из нашей практики (классификация стоматологических жалоб):**

Рассмотрим два сценария для текста *«Зуб болит при накусывании, десна опухла»* (правильный класс — периодонтит):

**Сценарий А (малая ошибка):**
- Пульпит: 5%
- Периодонтит: 85%
- Кариес: 5%
- Гингивит: 5%

Loss будет низким (≈0.16), так как модель присвоила высокую вероятность правильному классу.

**Сценарий Б (большая ошибка):**
- Пульпит: 40%
- Периодонтит: 10%
- Кариес: 30%
- Гингивит: 20%

Loss будет высоким (≈2.3), так как правильный класс получил низкую вероятность.

**Почему loss, а не accuracy?** Accuracy показывает только долю правильных ответов, но не учитывает **уверенность** модели. Loss чувствителен к уверенности: модель, которая даёт 80% на правильный класс, получает меньший loss, чем модель, дающая 51%, даже если обе «правильны». Это позволяет модели обучаться более плавно [3].

**Связь с Практикой 6:** на графиках мы видели, как **loss снижается** в процессе обучения. Это значит, что модель постепенно учится давать правильные ответы — ошибка становится меньше с каждой эпохой.

---

### 2.4. Градиентный спуск (Gradient Descent): механизм коррекции ошибок

**Определение:** градиентный спуск — это итеративный алгоритм оптимизации, который вычисляет **направление наискорейшего уменьшения** функции потерь и обновляет параметры модели в этом направлении [1], [2].

**Интуитивная аналогия (адаптирована для врачей):**

> Представьте, что вы оказались в горах в густом тумане и хотите спуститься в долину (где ошибка минимальна). Вы не видите всю гору, но можете оценить **локальный уклон** под ногами. Вы делаете шаг в сторону самого крутого спуска. Затем снова оцениваете уклон и делаете следующий шаг. Постепенно вы достигаете долины.
>
> **Соответствие:**
> - Гора = пространство всех возможных значений параметров модели.
> - Долина = набор параметров, при которых ошибка минимальна.
> - Уклон = градиент (производная), показывающий направление уменьшения ошибки.
> - Шаг = обновление параметров (размер шага контролируется learning rate).

**Математическая суть (без формул):**

Градиент — это вектор, указывающий направление **наибольшего возрастания** функции. Чтобы уменьшить функцию, мы двигаемся в **противоположном направлении**. Модель вычисляет градиент (через алгоритм обратного распространения ошибки) и обновляет параметры, сдвигая их на небольшую величину в сторону, противоположную градиенту.

**Роль learning rate (скорости обучения):**

Learning rate — это гиперпараметр, который определяет **размер шага** при обновлении параметров [4].

| Learning rate | Размер шага | Эффект | Аналогия |
|---------------|-------------|--------|----------|
| **Слишком малый (1e-5)** | Очень маленький | Медленная сходимость, но стабильная | Идти мелкими шагами — дойдёте, но долго |
| **Оптимальный (2e-5)** | Умеренный | Быстрая и стабильная сходимость | Идти уверенным шагом — оптимально |
| **Слишком большой (5e-5)** | Большой | Нестабильность, возможен «перескок» минимума | Прыгать с камня на камень — можно упасть |

**Эмпирическое наблюдение:** согласно исследованиям [6], для дообучения BERT-подобных моделей оптимальным значением learning rate является диапазон 2e-5 — 5e-5. Это значение используется в Практике 6.

---

### 2.5. Схема: один шаг обучения

```mermaid
flowchart TD
    subgraph Вход[📝 Итерация обучения]
        A[1️⃣ Подача примера<br>текст + правильный диагноз]
        B[2️⃣ Прямой проход<br>модель вычисляет предсказание]
        C[3️⃣ Вычисление loss<br>оценка ошибки]
        D[4️⃣ Обратный проход<br>вычисление градиента]
        E[5️⃣ Обновление параметров<br>шаг в сторону уменьшения ошибки]
    end
    
    A --> B --> C --> D --> E
    
    subgraph Повтор[🔄 Повторение]
        F[Следующий пример]
    end
    
    E --> F
    
    style Вход fill:#e1f5fe,stroke:#01579b
    style Повтор fill:#fff9c4,stroke:#f57f17
```

**Пошаговое пояснение:**

1. **Подача примера:** Модель получает текст и правильный диагноз. Это как если бы мы дали ординатору клинический случай и правильный ответ.

2. **Прямой проход:** Модель пропускает текст через все свои слои и выдаёт вероятности для каждого класса. Ординатор предлагает свою диагностическую гипотезу.

3. **Вычисление loss:** Сравниваем предсказание с правильным ответом. Чем больше разница, тем больше loss. Заведующий оценивает, насколько ординатор ошибся.

4. **Обратный проход:** Модель вычисляет градиент — направление, в котором нужно изменить параметры, чтобы уменьшить ошибку. Ординатор понимает, в каком направлении ему нужно корректировать свои знания.

5. **Обновление параметров:** Модель делает шаг в сторону уменьшения ошибки. Ординатор меняет свой подход на основе обратной связи.

6. **Повторение:** Берём следующий пример и повторяем всё сначала.

**Этот процесс повторяется тысячи раз, пока ошибка не перестанет уменьшаться.**

---

### 2.6. Основные выводы раздела

1. Обучение модели — это процесс настройки миллионов параметров с целью минимизации ошибки на обучающих данных.

2. Функция потерь (loss) количественно оценивает расхождение между предсказанием модели и правильным ответом. Снижение loss в процессе обучения является индикатором того, что модель учится.

3. Градиентный спуск — это алгоритм, который вычисляет направление уменьшения ошибки и обновляет параметры в этом направлении.

4. Скорость обучения (learning rate) контролирует размер шага при обновлении параметров и является критическим гиперпараметром.

5. Понимание этих механизмов позволяет клиницисту критически оценивать качество и ограничения ИИ-систем [5], [7].

---

## 3. ПАРАМЕТРЫ VS ГИПЕРПАРАМЕТРЫ

### 3.1. Параметры (Parameters): что модель учит сама

**Определение:** параметры модели — это внутренние переменные, значения которых **определяются в процессе обучения** на основе данных. Модель «учит» их сама, адаптируясь к закономерностям в обучающей выборке [1], [2].

**Техническое уточнение:** в нейросетях параметрами являются веса связей (weights) и смещения (biases). В модели ruBERT, используемой в Практике 6, их **178 миллионов**.

**Интуитивная аналогия:**

> Представьте, что ординатор учится на клиническом разборе. Он **меняет свой подход** к каждому пациенту на основе опыта: запоминает, какие симптомы с какими заболеваниями связаны, учится замечать паттерны, корректирует свои диагностические критерии. Это его «параметры» — они меняются в процессе обучения.

**Связь с Практикой 6:** когда мы запускали `trainer.train()`, именно эти 178 миллионов чисел **подстраивались** под нашу задачу классификации стоматологических жалоб.

---

### 3.2. Гиперпараметры (Hyperparameters): что задаём мы

**Определение:** гиперпараметры — это настройки процесса обучения, которые **задаются исследователем до начала обучения** и не изменяются в процессе [3], [4]. Они определяют, **как** модель будет учиться.

**Интуитивная аналогия:**

> Представьте, что ординатор идёт на цикл усовершенствования. Гиперпараметры — это **условия обучения**:
> - Сколько недель длится цикл (число эпох).
> - Как быстро ординатор должен усваивать новый материал (скорость обучения).
> - Сколько пациентов он осматривает за день (размер батча).
>
> Ординатор не выбирает эти условия — их выбирает руководитель (мы). Но они сильно влияют на результат.

**Главные гиперпараметры и их влияние:**

| Гиперпараметр | Что это | Аналогия для врача | Эффект при изменении |
|---------------|---------|---------------------|---------------------|
| **Learning Rate** | Скорость обучения. Насколько сильно меняются параметры на каждом шаге | Как быстро ординатор меняет подход после разбора ошибки | Слишком большой — «перепрыгивает» оптимум. Слишком маленький — учится очень медленно |
| **Number of Epochs** | Число эпох. Сколько раз модель видит все данные | Сколько раз ординатор повторяет цикл лекций и клинических разборов | Слишком мало — недообучение. Слишком много — переобучение |
| **Batch Size** | Сколько примеров за один шаг | Сколько клинических случаев разбирается за один раз | Маленький — быстрее, но «шумнее». Большой — стабильнее, но медленнее |
| **Weight Decay** | Регуляризация. Штраф за сложность модели | Ограничение: ординатор не должен усложнять интерпретацию без нужды | Большой — модель проще, меньше переобучается |

---

### 3.3. Схема: параметры vs гиперпараметры

```mermaid
flowchart LR
    subgraph Параметры["⚙️ Параметры (учит модель)"]
        P1["Веса связей<br>между нейронами"]
        P2["Смещения (bias)"]
        P3["178 млн чисел<br>в ruBERT"]
    end
    
    subgraph Гиперпараметры["🎛️ Гиперпараметры (задаём мы)"]
        H1["Learning Rate<br>скорость обучения"]
        H2["Number of Epochs<br>число эпох"]
        H3["Batch Size<br>размер батча"]
        H4["Weight Decay<br>регуляризация"]
    end
    
    subgraph Аналогия["🧠 Аналогия"]
        A1["Знания ординатора<br>меняются в процессе"]
        A2["Условия обучения<br>задаёт руководитель"]
    end
    
    Параметры --> Аналогия
    Гиперпараметры --> Аналогия
    
    style Параметры fill:#e1f5fe,stroke:#01579b
    style Гиперпараметры fill:#fff9c4,stroke:#f57f17
    style Аналогия fill:#c8e6c9,stroke:#2e7d32
```

---

### 3.4. Почему это важно для врача

- **Параметры** — это то, что модель выучила сама. Мы не можем их контролировать напрямую.
- **Гиперпараметры** — это то, что мы контролируем. От их выбора зависит качество обучения.
- Понимание гиперпараметров позволяет **критически оценивать** исследования: если авторы не указывают гиперпараметры, результаты сложно воспроизвести [7].

**Связь с Практикой 6:** мы проводили эксперименты, меняя `learning_rate` и `num_train_epochs`, и наблюдали, как это влияет на accuracy.

---

## 4. РАЗДЕЛЕНИЕ ДАННЫХ: TRAIN, VALIDATION, TEST

### 4.1. Проблема переобучения и необходимость разделения

**Определение:** переобучение (overfitting) — это ситуация, при которой модель запоминает обучающие примеры, но не способна обобщать закономерности на новые, невидимые ранее данные [3]. Это одна из самых распространённых ошибок в машинном обучении, которая приводит к завышенной оценке качества модели на обучающих данных и низкой эффективности в реальной клинической практике.

**Интуитивная аналогия из медицинского образования:**

> Представьте, что ординатор готовится к экзамену по диагностике кариеса. Ему дают 100 рентгеновских снимков с диагнозами для изучения. Он заучивает их наизусть — запоминает, какой диагноз соответствует какому снимку. На экзамене ему дают те же 100 снимков — он отвечает идеально. Но когда он приходит в клинику к реальному пациенту с новым снимком — он теряется. Он выучил конкретные случаи, но не понял общих рентгенологических признаков.
>
> В машинном обучении это называется **переобучением**. Модель «запоминает» обучающие примеры, но не учится «обобщать» — выделять закономерности, применимые к новым данным.

**Решение проблемы:** чтобы избежать переобучения и получить объективную оценку качества модели, данные разделяют на три независимые выборки: обучающую (train), валидационную (validation) и тестовую (test) [4].

---

### 4.2. Три типа выборок: назначение и функции

Каждая из трёх выборок выполняет свою уникальную функцию в процессе обучения. Понимание этих функций критически важно для корректной оценки модели.

| Выборка | Назначение | Аналогия в медицине | Особенности использования |
|---------|------------|----------------------|--------------------------|
| **Train** (обучающая) | Модель учится на этих данных. Только на них модель «видит» правильные ответы | Учебные клинические случаи, которые ординатор разбирает с наставником | Модель подстраивает параметры, минимизируя ошибку на этих данных |
| **Validation** (валидационная) | Настройка гиперпараметров и ранняя остановка обучения. Модель **не учится** на этих данных | Контрольные тесты, показывающие уровень усвоения материала | Мы оцениваем ошибку модели, но не обновляем параметры. Используется для выбора гиперпараметров |
| **Test** (тестовая) | **Однократная** финальная оценка качества модели. Модель никогда не видела эти данные | Финальный экзамен по новым, незнакомым случаям | Используется **только один раз** в конце обучения для получения объективной оценки |

**Ключевое методологическое правило:** тестовая выборка должна использоваться **строго один раз** — для финальной оценки модели. Любое использование тестовых данных для настройки модели (выбора гиперпараметров, ранней остановки) делает оценку необъективной [4].

**Типичное распределение данных:** в практике машинного обучения принято распределять данные в пропорции 70/15/15 или 80/10/10 [3]. Выбор пропорции зависит от объёма данных: чем больше данных, тем меньшую долю можно выделить на валидацию и тест. При малом объёме данных (менее 1000 примеров) долю валидационной и тестовой выборок можно увеличить до 20–25% каждая.

**Почему именно такие пропорции?**
- **Train (70–80%)** — достаточно данных, чтобы модель увидела разнообразие паттернов.
- **Validation (10–15%)** — достаточно данных, чтобы надёжно оценить качество настройки гиперпараметров.
- **Test (10–15%)** — достаточно данных, чтобы получить статистически значимую оценку финального качества.

---

### 4.3. Утечка данных (Data Leakage): определение и предотвращение

**Определение:** утечка данных (data leakage) — это ситуация, при которой информация из тестовой или валидационной выборки «просачивается» в процесс обучения модели. Это приводит к **необъективной** оценке качества модели, которая не будет соответствовать её реальной эффективности на новых данных [4].

**Интуитивная аналогия:**

> Представьте, что вы готовите ординатора к экзамену. Утечка данных — это когда вы даёте ему **те же самые вопросы**, которые будут на экзамене. Он выучит их наизусть, но не поймёт предмет. В реальной практике он не сможет применить знания.

**Типичные ошибки, приводящие к утечке данных:**

| Ошибка | Почему это утечка | Правильное решение |
|--------|-------------------|-------------------|
| **Нормализация всех данных перед разделением** | Параметры нормализации (среднее, стандартное отклонение) вычислены с учётом тестовых данных | Сначала разделить данные, затем нормализовать **только обучающую** выборку, а параметры нормализации применить к валидационной и тестовой |
| **Использование тестовой выборки для настройки гиперпараметров** | Тестовая выборка перестаёт быть «независимой» — модель «подглядывает» ответы | Все настройки выполняются **только** через валидационную выборку. Тест используется **один раз** в конце |
| **Перемешивание данных без учёта времени** | При использовании временных данных (например, записи пациентов за несколько лет) случайное перемешивание может привести к тому, что модель «узнает» будущее | Делить по времени: train — более ранние данные, test — более поздние |

**Стратификация (stratification):** при разделении данных важно сохранять пропорцию классов в каждой выборке. Например, если в общей выборке 20% пациентов с периодонтитом, то и в обучающей, и в валидационной, и в тестовой выборках должно быть примерно 20% пациентов с периодонтитом [4]. В противном случае модель может «не научиться» распознавать редкий класс. Стратификация особенно важна в клинических исследованиях, где целевые состояния (осложнения, редкие заболевания) встречаются относительно редко.

---

### 4.4. Схема: правильное разделение данных

На схеме ниже показан полный процесс разделения данных от исходного датасета до финальной оценки модели.

```mermaid
flowchart LR
    subgraph Данные[📊 Все данные]
        D[Исходный датасет]
    end
    
    subgraph Разделение[✂️ Разделение]
        T[🚂 Train<br>70%]
        V[🛤️ Validation<br>15%]
        Te[🧪 Test<br>15%]
    end
    
    subgraph Процесс[⚙️ Процесс]
        P1[Модель учится<br>на этих данных]
        P2[Настройка<br>гиперпараметров]
        P3[Финальная<br>оценка]
    end
    
    Данные --> Разделение
    T --> P1
    V --> P2
    Te --> P3
    
    style T fill:#c8e6c9,stroke:#2e7d32
    style V fill:#fff9c4,stroke:#f57f17
    style Te fill:#ffcdd2,stroke:#c62828
```

**Пояснение к схеме:**
- **Train (70%)** — основной объём данных, на котором модель обучается. Модель видит правильные ответы и подстраивает параметры.
- **Validation (15%)** — используется для настройки гиперпараметров и ранней остановки. Модель не учится на этих данных, но мы отслеживаем её ошибку.
- **Test (15%)** — используется **один раз в конце** для финальной оценки качества модели. Модель никогда не видела эти данные.

---

### 4.5. Связь с Практикой 6

В Практике 6 мы использовали датасет, где данные уже были предварительно разделены. Это стандартная практика в библиотеке Hugging Face Datasets — многие датасеты поставляются с готовым разделением на train, validation и test.

**Конкретные цифры из Практики 6:**
- `train` — 10 000 примеров (обучение)
- `validation` — 2000 примеров (настройка гиперпараметров)
- `test` — 2000 примеров (финальная проверка)

Это разделение соответствовало принципам стратификации — пропорции классов в каждой выборке были сохранены. Именно благодаря этому мы могли доверять результатам оценки модели на тестовой выборке. Если бы данные были разделены неправильно (например, без стратификации или с утечкой данных), мы не могли бы утверждать, что модель действительно работает хорошо.

---

### 4.6. Практический пример: создание синтетического датасета для обучения

Для того чтобы увидеть процесс разделения данных в действии, мы создадим **синтетический датасет** — искусственные тексты, имитирующие обращения пациентов с разными стоматологическими проблемами.

**Зачем создавать синтетические данные?**

В клинической медицине мы не можем использовать реальные данные пациентов без специального разрешения — это нарушает принципы конфиденциальности и врачебной тайны [7]. Однако для обучения и экспериментов нам нужны данные, которые отражают реальные клинические сценарии.

**Решение:** создание **синтетических (искусственных) данных** — текстов, которые имитируют реальные жалобы пациентов, но не принадлежат конкретным людям [5]. Это позволяет:
- Обучать модели без риска утечки конфиденциальной информации.
- Экспериментировать с разными параметрами обучения.
- Демонстрировать принципы работы моделей на понятных примерах.

---

#### 4.6.1. Структура синтетического датасета

Мы создадим датасет из текстов, размеченных по четырём клиническим классам. Каждый класс соответствует определённому состоянию, с которым стоматолог может столкнуться в практике.

| Класс | Диагноз | Примеры текстов | Клинический контекст |
|-------|---------|-----------------|---------------------|
| 0 | Кариес | «Заметил тёмное пятно на зубе, иногда реагирует на сладкое» | Поверхностный или средний кариес, плановая санация |
| 1 | Пульпит | «Острая боль, особенно ночью, от холодного и горячего, обезболивающее почти не помогает» | Воспаление пульпы, требуется эндодонтическое лечение |
| 2 | Периодонтит | «Боль при накусывании, десна опухла, щека припухла, температура 37.5» | Воспаление периодонта, возможно, периостит |
| 3 | Гингивит | «Дёсны кровоточат при чистке зубов, неприятный запах изо рта, отёк десневого края» | Воспаление десны, требуется профессиональная гигиена |

---

#### 4.6.2. Код для создания синтетического датасета

**Пояснение к коду:** приведённый ниже код создаёт датасет из 120 примеров (по 30 на каждый класс). Для каждого класса определён список типичных фраз, из которых случайным образом выбираются тексты. Это имитирует разнообразие жалоб, которые врач может встретить в реальной практике.

```python
# Ячейка 1: Создание синтетического датасета стоматологических жалоб

import pandas as pd
import random

# Определяем словари фраз для каждого класса
diagnosis_phrases = {
    0: [  # Кариес
        "Заметил тёмное пятно на зубе, иногда реагирует на сладкое.",
        "Появилась небольшая дырка в зубе, боли нет.",
        "Зуб стал чувствительным к холодному, но быстро проходит.",
        "Между зубами застревает пища, есть коричневое пятно.",
        "При осмотре обнаружили кариес на боковой поверхности.",
        "Зуб немного потемнел, но не болит.",
        "Иногда чувствую сладкое, но быстро проходит.",
        "На переднем зубе появилось белое пятно, шероховатое.",
        "Обнаружил кариес на жевательной поверхности, требуется пломба.",
        "Зуб реагирует на холодное, но не сильно."
    ],
    1: [  # Пульпит
        "Острая боль в зубе, особенно ночью, от холодного и горячего.",
        "Зуб болит сам по себе, обезболивающее почти не помогает.",
        "Сильная пульсирующая боль, отдаёт в ухо или висок.",
        "Боль появляется приступами, длится несколько минут.",
        "Ночью просыпаюсь от боли в зубе, не могу спать.",
        "Зуб реагирует на горячее, боль долго не проходит.",
        "Боль в зубе не проходит после приёма анальгетиков.",
        "Чувствую, как будто зуб «дёргает», больно при накусывании.",
        "Зуб болит уже несколько дней, терпеть невозможно.",
        "Боль в зубе отдаёт в челюсть, не могу определить какой именно зуб."
    ],
    2: [  # Периодонтит
        "Боль при накусывании, десна опухла, щека припухла, температура 37.5.",
        "Зуб болит при надавливании, десна красная и отёчная.",
        "После лечения зуб стал болеть при жевании, появился отёк.",
        "Десна около зуба опухла, есть гнойный привкус во рту.",
        "Зуб как будто «вырос», прикус не смыкается из-за отёка.",
        "Боль в зубе постоянная, усиливающаяся при любом прикосновении.",
        "При надавливании на зуб чувствую резкую боль, щека отекла.",
        "Температура 37.8, зуб болит, десна над ним горячая.",
        "После переохлаждения появилась боль в зубе и отёк щеки.",
        "Зуб ранее лечили, теперь он болит и шатается, десна воспалена."
    ],
    3: [  # Гингивит
        "Дёсны кровоточат при чистке зубов, неприятный запах изо рта.",
        "Десневой край отёк и покраснел, боль при прикосновении.",
        "Кровоточивость дёсен уже несколько недель, особенно утром.",
        "Дёсны стали рыхлыми, легко повреждаются.",
        "Неприятный запах изо рта, дёсны кровоточат даже без чистки.",
        "Десна опухла и болит, но зуб не беспокоит.",
        "При чистке зубов сплёвываю кровь, дёсны гиперемированы.",
        "Дёсны отёкшие, есть налёт, зубной камень.",
        "Зуд и жжение в дёснах, они стали синюшными.",
        "После снятия зубного камня дёсны продолжают кровоточить."
    ]
}

def create_dataset(num_samples_per_class=25):
    """
    Создаёт синтетический датасет стоматологических жалоб.
    
    Параметры:
        num_samples_per_class: количество примеров для каждого класса
    
    Возвращает:
        pd.DataFrame с колонками 'text' и 'label'
    """
    data = []
    for label, phrases in diagnosis_phrases.items():
        for _ in range(num_samples_per_class):
            text = random.choice(phrases)
            # Добавляем небольшие вариации для естественности
            if random.random() > 0.6:
                text = text.replace(".", "...")
            if random.random() > 0.8:
                text = text + " (очень беспокоит)"
            data.append({
                "text": text,
                "label": label
            })
    random.shuffle(data)  # Перемешиваем для предотвращения систематических ошибок
    return pd.DataFrame(data)

# Создаём датасет из 120 примеров (30 на каждый класс)
df = create_dataset(num_samples_per_class=30)

print(f"✅ Создано {len(df)} примеров")
print("\nПервые 10 примеров:")
print(df.head(10))

# Проверяем распределение классов
print("\n📊 Распределение классов:")
diagnosis_names = {0: "Кариес", 1: "Пульпит", 2: "Периодонтит", 3: "Гингивит"}
for label, count in df['label'].value_counts().sort_index().items():
    print(f"  {diagnosis_names[label]}: {count} примеров")
```

**Ожидаемый вывод:**

```
✅ Создано 120 примеров

Первые 10 примеров:
                                              text  label
0             Дёсны кровоточат при чистке зубов...      3
1                     Острая боль в зубе, особенно ночью.      1
2                       Зуб стал чувствительным к холодному.      0
3      Боль при накусывании, десна опухла, щека припухла.      2
4                        Дёсны стали рыхлыми, легко повреждаются.      3
5                     Ночью просыпаюсь от боли в зубе.      1
6               Между зубами застревает пища, есть коричневое пятно.      0
7      После переохлаждения появилась боль в зубе и отёк щеки.      2
8            Зуб реагирует на горячее, боль долго не проходит.      1
9            На переднем зубе появилось белое пятно, шероховатое.      0

📊 Распределение классов:
  Кариес: 30 примеров
  Пульпит: 30 примеров
  Периодонтит: 30 примеров
  Гингивит: 30 примеров
```

---

#### 4.6.3. Визуализация распределения классов

```python
# Ячейка 2: Визуализация распределения классов

import matplotlib.pyplot as plt

counts = df['label'].value_counts().sort_index()
diagnosis_names = {0: "Кариес", 1: "Пульпит", 2: "Периодонтит", 3: "Гингивит"}

plt.figure(figsize=(8, 5))
bars = plt.bar([diagnosis_names[i] for i in counts.index],
               counts.values,
               color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])

plt.title('Распределение диагнозов в синтетическом датасете', fontsize=14)
plt.xlabel('Диагноз', fontsize=12)
plt.ylabel('Количество примеров', fontsize=12)

for bar, count in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', va='bottom', fontsize=12)

plt.grid(axis='y', alpha=0.3)
plt.show()
```

---

#### 4.6.4. Разделение данных: демонстрация

```python
# Ячейка 3: Разделение данных на train, validation, test

from sklearn.model_selection import train_test_split

X = df['text'].values
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("📊 Разделение данных:")
print(f"  Train:     {len(X_train)} примеров ({len(X_train)/len(df)*100:.0f}%)")
print(f"  Validation: {len(X_val)} примеров ({len(X_val)/len(df)*100:.0f}%)")
print(f"  Test:      {len(X_test)} примеров ({len(X_test)/len(df)*100:.0f}%)")

print("\n📊 Распределение классов в каждой выборке:")
for name, y_data in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"\n  {name}:")
    for label in sorted(set(y_data)):
        count = sum(y_data == label)
        pct = count / len(y_data) * 100
        print(f"    {diagnosis_names[label]}: {count} ({pct:.1f}%)")
```

---

#### 4.6.5. Сохранение данных

```python
# Ячейка 4: Сохранение данных

df_splits = pd.DataFrame({
    'text': list(X_train) + list(X_val) + list(X_test),
    'label': list(y_train) + list(y_val) + list(y_test),
    'split': ['train']*len(X_train) + ['val']*len(X_val) + ['test']*len(X_test)
})

df_splits.to_csv("dental_diagnosis_dataset_splits.csv", index=False)
print("✅ Датасет с разделением сохранён как 'dental_diagnosis_dataset_splits.csv'")
```

---

#### 4.6.6. Схема: от синтетических данных к обучению

```mermaid
flowchart LR
    A[📝 Синтетические<br>жалобы пациентов] --> B[✂️ Разделение<br>70/15/15]
    B --> C[🚂 Train<br>обучение]
    B --> D[🛤️ Validation<br>настройка]
    B --> E[🧪 Test<br>финальная оценка]
    
    C --> F[🧠 Дообучение<br>BERT]
    F --> G[📊 Графики<br>обучения]
    G --> H[✅ Готовая<br>модель]
    
    style A fill:#e1f5fe,stroke:#01579b
    style B fill:#fff9c4,stroke:#f57f17
    style C fill:#c8e6c9,stroke:#2e7d32
    style D fill:#fff9c4,stroke:#f57f17
    style E fill:#ffcdd2,stroke:#c62828
    style H fill:#c8e6c9,stroke:#2e7d32
```

---

### 4.7. Основные выводы раздела

1. **Разделение данных на обучающую, валидационную и тестовую выборки — это обязательная практика** для получения объективной оценки качества модели [4].

2. **Утечка данных** — одна из самых опасных ошибок, которая делает оценку модели необъективной.

3. **Стратификация** особенно важна для клинических данных, где редкие состояния (осложнения, неотложные случаи) требуют особого внимания.

4. **Синтетические данные** — это этичный и практичный способ создания датасетов для обучения, который не нарушает конфиденциальность пациентов [5].

5. Понимание принципов разделения данных позволяет врачу **критически оценивать исследования**.

---

## 5. ГРАФИКИ ОБУЧЕНИЯ: КАК ЧИТАТЬ КРИВЫЕ LOSS И ACCURACY

### 5.1. Что показывают графики обучения и зачем они нужны

**Определение:** кривые обучения (learning curves) — это графики, которые показывают динамику изменения ошибки (loss) и точности (accuracy) модели в процессе обучения на обучающей (train) и валидационной (validation) выборках [6].

Они позволяют ответить на вопросы:
- Учится ли модель? (снижается ли loss?)
- Не переобучается ли модель? (не расходится ли train и val loss?)
- Не недоучилась ли модель? (достаточно ли низкий loss?)
- Когда нужно остановить обучение? (когда val loss перестаёт снижаться)

**Интуитивная аналогия из клинического обучения:**

> Представьте, что вы — наставник ординатора. Каждую неделю вы даёте ему два типа заданий:
> 1. **Учебные кейсы** (train) — те, которые он уже разбирал. Он должен показывать хорошие результаты.
> 2. **Новые кейсы** (validation) — те, которые он видит впервые. Они показывают, научился ли он обобщать знания.
>
> Если ординатор отлично справляется с учебными кейсами, но проваливает новые — он просто запомнил ответы (переобучение). Если он плохо справляется и с теми, и с другими — он недостаточно учился (недообучение). Если он хорошо справляется с обоими типами — он учится правильно.

| График | Что показывает | Что означает снижение/рост |
|--------|----------------|---------------------------|
| **Loss (ошибка)** | Насколько сильно модель ошибается | Снижение = модель учится. Рост = проблемы |
| **Accuracy (точность)** | Доля правильных ответов | Рост = модель становится точнее. Падение = проблемы |

**Ключевой принцип:** важно смотреть **не только на финальные значения**, но и на **соотношение** train и validation кривых. Разрыв между кривыми — главный диагностический признак.

---

### 5.2. Идеальное обучение

Характеристики: обе кривые (train и val) плавно идут вниз (loss) и вверх (accuracy), разрыв минимален.

**Код для визуализации идеального обучения:** (аналогичный, но с другими числами, например, loss от 2.0 до 0.3)

```python
# Ячейка 1: Генерация графиков идеального обучения

import numpy as np
import matplotlib.pyplot as plt

epochs = np.arange(1, 21)

train_loss = 2.0 * np.exp(-epochs / 5) + 0.3
val_loss = 2.2 * np.exp(-epochs / 5) + 0.35
train_acc = 0.92 - 0.62 * np.exp(-epochs / 4)
val_acc = 0.90 - 0.60 * np.exp(-epochs / 4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Эпоха', fontsize=12)
ax1.set_ylabel('Loss (ошибка)', fontsize=12)
ax1.set_title('Идеальное обучение: Loss снижается', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Эпоха', fontsize=12)
ax2.set_ylabel('Accuracy (точность)', fontsize=12)
ax2.set_title('Идеальное обучение: Accuracy растёт', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()
```

---

### 5.3. Переобучение (Overfitting)

Характеристики: train loss продолжает снижаться, а val loss начинает **расти**. Train accuracy растёт до 95%+, а val accuracy **падает**.

**Код для визуализации переобучения:**

```python
# Ячейка 2: Генерация графиков переобучения

epochs = np.arange(1, 21)

train_loss = 2.0 * np.exp(-epochs / 3) + 0.05
val_loss = 2.2 * np.exp(-epochs / 4) + 0.1 + 0.02 * (epochs - 10) * (epochs > 10) * 10
train_acc = 0.98 - 0.68 * np.exp(-epochs / 3)
val_acc = 0.90 - 0.60 * np.exp(-epochs / 4) - 0.015 * (epochs - 10) * (epochs > 10) * 10

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
ax1.axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='Переобучение')
ax1.set_xlabel('Эпоха', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('⚠️ Переобучение: Val Loss растёт!', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
ax2.axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='Переобучение')
ax2.set_xlabel('Эпоха', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('⚠️ Переобучение: Val Accuracy падает!', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()
```

**Что делать при переобучении:** ранняя остановка, уменьшение learning rate, увеличение weight decay, аугментация данных, dropout.

---

### 5.4. Недообучение (Underfitting)

Характеристики: обе кривые остаются высокими (loss) и низкими (accuracy), улучшения незначительные.

**Код для визуализации недообучения:**

```python
# Ячейка 3: Генерация графиков недообучения

epochs = np.arange(1, 21)

train_loss = 2.0 - 0.01 * epochs
val_loss = 2.1 - 0.01 * epochs
train_acc = 0.35 + 0.005 * epochs
val_acc = 0.33 + 0.005 * epochs

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Эпоха', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('⚠️ Недообучение: Loss высокий', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Эпоха', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('⚠️ Недообучение: Accuracy низкий', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()
```

**Что делать при недообучении:** увеличить число эпох, увеличить learning rate, усложнить модель, проверить данные.

---

### 5.5. Практический код для сравнения всех трёх сценариев

```python
# Ячейка 4: Сравнение всех трёх сценариев

def generate_learning_curves(scenario='ideal'):
    epochs = np.arange(1, 21)
    if scenario == 'ideal':
        train_loss = 2.0 * np.exp(-epochs / 5) + 0.3
        val_loss = 2.2 * np.exp(-epochs / 5) + 0.35
        train_acc = 0.92 - 0.62 * np.exp(-epochs / 4)
        val_acc = 0.90 - 0.60 * np.exp(-epochs / 4)
    elif scenario == 'overfitting':
        train_loss = 2.0 * np.exp(-epochs / 3) + 0.05
        val_loss = 2.2 * np.exp(-epochs / 4) + 0.1 + 0.02 * (epochs - 10) * (epochs > 10) * 10
        train_acc = 0.98 - 0.68 * np.exp(-epochs / 3)
        val_acc = 0.90 - 0.60 * np.exp(-epochs / 4) - 0.015 * (epochs - 10) * (epochs > 10) * 10
    else:  # underfitting
        train_loss = 2.0 - 0.01 * epochs
        val_loss = 2.1 - 0.01 * epochs
        train_acc = 0.35 + 0.005 * epochs
        val_acc = 0.33 + 0.005 * epochs
    return epochs, train_loss, val_loss, train_acc, val_acc

scenarios = {
    'Идеальное обучение ✅': 'ideal',
    'Переобучение ⚠️': 'overfitting',
    'Недообучение ⚠️': 'underfitting'
}

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for row, (title, scenario) in enumerate(scenarios.items()):
    epochs, train_loss, val_loss, train_acc, val_acc = generate_learning_curves(scenario)
    
    ax_loss = axes[row, 0]
    ax_loss.plot(epochs, train_loss, 'b-', label='Train Loss', linewidth=2)
    ax_loss.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)
    ax_loss.set_xlabel('Эпоха', fontsize=11)
    ax_loss.set_ylabel('Loss', fontsize=11)
    ax_loss.set_title(f'{title} - Loss', fontsize=13)
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)
    ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.2)
    
    ax_acc = axes[row, 1]
    ax_acc.plot(epochs, train_acc, 'b-', label='Train Accuracy', linewidth=2)
    ax_acc.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)
    ax_acc.set_xlabel('Эпоха', fontsize=11)
    ax_acc.set_ylabel('Accuracy', fontsize=11)
    ax_acc.set_title(f'{title} - Accuracy', fontsize=13)
    ax_acc.legend()
    ax_acc.grid(True, alpha=0.3)
    ax_acc.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()
```

---

### 5.6. Таблица: как распознать состояние обучения

| Что видим на графиках | Состояние | Что делать | Клиническая аналогия |
|-----------------------|-----------|------------|---------------------|
| Train loss ↓, Val loss ↓, разрыв мал | ✅ Идеальное обучение | Продолжать, остановиться при плато | Ординатор успешно учится |
| Train loss ↓, Val loss ↑, разрыв растёт | ⚠️ Переобучение | Уменьшить эпохи, уменьшить LR, добавить регуляризацию | Ординатор заучил случаи, не может обобщать |
| Оба loss высокие, не падают | ⚠️ Недообучение | Увеличить эпохи, увеличить LR, усложнить модель | Ординатор слишком мало учился |
| Train loss ↓, Val loss застыл | Модель перестала учиться | Увеличить LR, проверить данные | Ординатор достиг «плато», нужен новый подход |

---

### 5.7. Основные выводы раздела

Понимание графиков обучения — ключ к оценке качества модели. Врач, видя кривые, может судить о переобучении, недообучении, правильности выбора гиперпараметров.

---

## 6. МЕТРИКИ ДЛЯ КЛАССИФИКАЦИИ

### 6.1. Зачем нужны метрики качества?

Метрики дают объективную оценку модели, позволяют сравнивать модели, выявлять слабые места. В стоматологии они критически важны: например, при скрининге неотложных состояний (острый периодонтит, абсцесс) нельзя пропустить ни одного пациента (высокий recall), а при направлении на дорогостоящее обследование важно не давать ложных направлений (высокая precision).

---

### 6.2. Почему accuracy не всегда лучшая метрика

**Проблема 1: Несбалансированные классы.**

Пример: модель для выявления острого периодонтита среди всех обращений. Пусть распространённость острого периодонтита 5% (50 из 1000). Модель, всегда предсказывающая «нет периодонтита», будет иметь accuracy 95%, но не выявит ни одного пациента. Это опасно.

**Проблема 2: Разная цена ошибок.** Пропустить острый абсцесс (FN) гораздо опаснее, чем ложно направить на осмотр (FP).

---

### 6.3. Матрица ошибок

| | Предсказано: Периодонтит | Предсказано: Нет |
|---|---|---|
| **На самом деле: Периодонтит** | TP | FN (пропустили) |
| **На самом деле: Нет** | FP (ложная тревога) | TN |

---

### 6.4. Precision

**Precision = TP / (TP + FP)** — из всех, кого модель назвала «периодонтит», сколько действительно имеют его. Важна, когда ложные тревоги дороги (например, направление на КТ, биопсию).

---

### 6.5. Recall

**Recall = TP / (TP + FN)** — из всех с реальным периодонтитом сколько найдено. Важна, когда нельзя пропустить (скрининг острого воспаления, суицидальный риск в анамнезе).

---

### 6.6. F1-score

**F1 = 2 * (Precision * Recall) / (Precision + Recall)** — баланс. Используется, когда нужен компромисс.

---

### 6.7. Сравнительная таблица метрик с клиническими примерами

| Метрика | Формула | Когда важна | Пример |
|---------|---------|-------------|--------|
| Accuracy | (TP+TN)/все | Сбалансированные классы | Исследовательские задачи |
| Precision | TP/(TP+FP) | Дорогие ложные тревоги | Направление на дорогостоящее обследование |
| Recall | TP/(TP+FN) | Нельзя пропустить | Скрининг острого периодонтита |
| F1 | Гармоническое среднее | Нужен баланс | Сравнение моделей |

---

### 6.8. Пример расчёта на скрининге острого периодонтита

Допустим, 1000 пациентов, 100 с острым периодонтитом (10%). Модель предсказывает:

| | Предсказано: Периодонтит | Предсказано: Нет |
|---|---|---|
| **На самом деле: Периодонтит** | TP=90 | FN=10 |
| **На самом деле: Нет** | FP=180 | TN=720 |

Accuracy = (90+720)/1000 = 81%
Precision = 90/(90+180) = 33.3%
Recall = 90/(90+10) = 90%
F1 = 2*(0.333*0.9)/(0.333+0.9) ≈ 0.49

Интерпретация: модель находит 90% больных (хороший recall), но много ложных тревог (низкая precision). Для скрининга это приемлемо, так как лучше перестраховаться, но нужны ресурсы для проверки всех положительных результатов.

---

### 6.9. Какую метрику выбрать в стоматологии: практические рекомендации

| Задача | Основная метрика | Целевое значение |
|--------|------------------|------------------|
| Скрининг острого периодонтита/абсцесса по жалобам | Recall | > 0.95 |
| Определение необходимости удаления зуба | Precision | > 0.90 |
| Оценка риска осложнений после имплантации | Recall | > 0.85 |
| Подбор тактики лечения (терапевтическое vs хирургическое) | Precision | > 0.85 |
| Исследовательские цели | F1 | > 0.75 |

---

### 6.10. Схема выбора метрики

```mermaid
flowchart TD
    Start[🎯 Какая клиническая задача?]
    Start --> Q1[Важнее не пропустить<br>опасное состояние?]
    Q1 -->|Да| Recall[🎯 Выбираем RECALL]
    Q1 -->|Нет| Q2[Важнее не ошибаться,<br>когда модель даёт прогноз?]
    Q2 -->|Да| Precision[🎯 Выбираем PRECISION]
    Q2 -->|Нет| F1[🎯 Выбираем F1]
    
    Recall --> R1[⚠️ Приоритет: не пропустить<br>ценой ложных тревог<br><br>Примеры:<br>✓ скрининг острого периодонтита<br>✓ выявление абсцесса<br>✓ оценка риска анафилаксии]
    Precision --> P1[⚠️ Приоритет: не ошибиться<br>в положительном прогнозе<br><br>Примеры:<br>✓ направление на КТ<br>✓ назначение антибиотиков<br>✓ решение об удалении]
    F1 --> F1_1[⚖️ Баланс между<br>точностью и полнотой<br><br>Примеры:<br>✓ исследовательские задачи<br>✓ сравнение моделей]
```

---

### 6.11. Код для расчёта метрик на синтетических данных

```python
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

np.random.seed(42)
n_samples = 1000
true_labels = np.array([0]*900 + [1]*100)  # 900 здоровых, 100 периодонтит
predicted_labels = np.array([0]*850 + [1]*50 + [1]*70 + [0]*30)

cm = confusion_matrix(true_labels, predicted_labels)
print("Матрица ошибок:\n", cm)

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print("\nОтчёт:\n", classification_report(true_labels, predicted_labels, target_names=['Здоров', 'Периодонтит']))
```

---

### 6.12. Основные выводы

Метрики нужно выбирать осознанно. Accuracy может обманывать при дисбалансе классов. В стоматологии для неотложных состояний важнее recall, для плановых направлений — precision.

---

## 7. СВЯЗЬ С ДООБУЧЕНИЕМ (FINE-TUNING)

### 7.1. Что такое дообучение и зачем оно нужно

Дообучение — адаптация предобученной модели (например, ruBERT) к конкретной задаче (классификация стоматологических жалоб). Модель уже знает русский язык, мы донастраиваем её на небольшом размеченном датасете.

**Интуитивная аналогия:** ординатор, изучивший общую медицину, проходит специализацию по стоматологии. Он использует базовые знания (анатомия, физиология), но учится специфическим навыкам (препарирование, рентгенодиагностика).

---

### 7.2. Что происходит при дообучении: пошаговый разбор

1. Берём предобученную модель (ruBERT) с 178 млн параметров.
2. Заменяем последний слой: вместо предсказания слов — классификация 4 диагнозов.
3. Обучаем с маленьким learning rate (2e-5), чтобы не разрушить общие языковые знания.
4. Следим за графиками обучения, используем early stopping.
5. Оцениваем на тестовой выборке метриками.

**Почему маленький learning rate?** Чтобы сохранить знания языка, полученные при предобучении. Если learning rate большой, модель «забудет» русский язык и переобучится на узкой задаче.

---

### 7.3. Схема процесса дообучения

```mermaid
flowchart TD
    A[📚 Предобученная модель<br>знает русский язык] --> B[🔧 Замена последнего слоя<br>на 4 класса]
    B --> C[⚙️ Дообучение на датасете<br>стоматологических жалоб<br>с малым learning rate]
    C --> D[📊 Мониторинг графиков]
    D --> E{Переобучилась?}
    E -->|Да| F[Ранняя остановка]
    E -->|Нет| G[Обучение завершено]
    F --> G
    G --> H[Оценка на тесте]
    H --> I[🎯 Готовая модель]
```

---

### 7.4. Практические рекомендации для дообучения в стоматологии

| Параметр | Рекомендация | Почему |
|----------|--------------|--------|
| Learning rate | 2e-5 — 5e-5 | Сохраняет общие языковые знания |
| Число эпох | 2–4 | Больше — риск переобучения |
| Batch size | 16–32 | Зависит от ресурсов |
| Weight decay | 0.01–0.05 | Предотвращает переобучение |
| Early stopping | Использовать | Останавливает обучение при росте val loss |
| Стратификация | Обязательно | Сохраняет пропорции диагнозов в выборках |

---

### 7.5. Этический аспект дообучения в стоматологии

Дообученная модель может быть предвзятой, если обучающие данные нерепрезентативны (например, только взрослые пациенты, только кариес). Перед внедрением нужно проверить модель на разных группах. Всегда сохраняется ответственность врача.

---

### 7.6. Полный код для дообучения на синтетическом датасете стоматологических жалоб

```python
# ============================================================
# 1. УСТАНОВКА И ИМПОРТ
# ============================================================
# !pip install transformers datasets scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Библиотеки загружены")

# ============================================================
# 2. СОЗДАНИЕ СИНТЕТИЧЕСКОГО ДАТАСЕТА
# ============================================================
diagnosis_phrases = {
    0: ["Заметил тёмное пятно на зубе...", ...],  # 10 фраз
    1: ["Острая боль в зубе...", ...],
    2: ["Боль при накусывании...", ...],
    3: ["Дёсны кровоточат...", ...]
}

def create_dataset(num_samples_per_class=30):
    data = []
    for label, phrases in diagnosis_phrases.items():
        for _ in range(num_samples_per_class):
            text = random.choice(phrases)
            data.append({"text": text, "label": label})
    random.shuffle(data)
    return pd.DataFrame(data)

df = create_dataset(30)
print(f"Создано {len(df)} примеров")

# ============================================================
# 3. РАЗДЕЛЕНИЕ ДАННЫХ
# ============================================================
X_train, X_temp, y_train, y_temp = train_test_split(df['text'], df['label'], test_size=0.3, random_state=42, stratify=df['label'])
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

dataset = DatasetDict({
    "train": Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()}),
    "validation": Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()}),
    "test": Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})
})

# ============================================================
# 4. ТОКЕНИЗАЦИЯ
# ============================================================
model_name = "DeepPavlov/rubert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# ============================================================
# 5. ЗАГРУЗКА МОДЕЛИ
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# ============================================================
# 6. НАСТРОЙКА ОБУЧЕНИЯ
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    dataloader_pin_memory=False,
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, average='weighted', zero_division=0),
        'recall': recall_score(labels, predictions, average='weighted', zero_division=0),
        'f1': f1_score(labels, predictions, average='weighted', zero_division=0)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ============================================================
# 7. ОБУЧЕНИЕ
# ============================================================
print("Начинаем обучение...")
trainer.train()
trainer.save_model("./dental_model")

# ============================================================
# 8. ОЦЕНКА НА ТЕСТЕ
# ============================================================
test_results = trainer.predict(tokenized_dataset["test"])
predictions = np.argmax(test_results.predictions, axis=1)
true_labels = test_results.label_ids

accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions, average='weighted', zero_division=0)
recall = recall_score(true_labels, predictions, average='weighted', zero_division=0)
f1 = f1_score(true_labels, predictions, average='weighted', zero_division=0)

print("\nРезультаты на тесте:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

print("\nОтчёт по классам:")
diagnosis_names = {0: "Кариес", 1: "Пульпит", 2: "Периодонтит", 3: "Гингивит"}
print(classification_report(true_labels, predictions, target_names=list(diagnosis_names.values()), zero_division=0))

# ============================================================
# 9. ГРАФИКИ ОБУЧЕНИЯ (можно добавить)
# ============================================================
# ... (аналогично, но с учётом логов)
```

---

## 8. ЗАКЛЮЧЕНИЕ

**Что мы сегодня узнали:**

| Что узнали | Ключевая идея | Аналогия для врача |
|------------|---------------|--------------------|
| Что такое обучение | Настройка параметров модели для уменьшения ошибки | Ординатор учится на разборах |
| Функция потерь | Измеряет ошибку | Оценка ошибки в диагнозе |
| Градиентный спуск | Алгоритм коррекции параметров | Спуск с горы в тумане |
| Параметры vs гиперпараметры | Параметры учит модель, гиперпараметры задаём мы | Знания ординатора vs условия обучения |
| Разделение данных | Train/validation/test | Учебные кейсы, контрольные, экзамен |
| Графики обучения | Распознавание переобучения и недообучения | Как понять, что ординатор заучивает |
| Метрики | Accuracy, precision, recall, F1 | В стоматологии recall важнее accuracy для неотложных состояний |

**Ключевая мысль:** Понимание процесса обучения — ключ к критической оценке ИИ-систем. Врач, понимающий, как модель учится, может обоснованно доверять или не доверять её прогнозам.

---

## 9. ВОПРОСЫ ДЛЯ САМОПРОВЕРКИ

1. Что такое функция потерь? Приведите аналогию из клинической практики.
2. Что такое градиентный спуск? Объясните на примере спуска с горы.
3. В чём разница между параметрами и гиперпараметрами? Приведите примеры тех и других для стоматологической модели.
4. Зачем делить данные на train, validation и test? Что произойдёт, если использовать test для настройки?
5. Что такое утечка данных? Приведите пример в стоматологическом исследовании.
6. Как выглядит график переобучения? Что делать?
7. Как выглядит график недообучения? Что делать?
8. Что такое precision и recall? Какая метрика важнее для скрининга острого периодонтита и почему?
9. Что такое F1-score и когда его используют?
10. Зачем создавать синтетические данные в стоматологии?
11. Что такое дообучение? Почему оно эффективно и какие гиперпараметры критичны?
12. **Этический вопрос:** модель для выявления острого абсцесса имеет precision=90%, recall=60%. Можно ли её использовать? Почему? Как улучшить?

---

## 10. ДОМАШНЕЕ ЗАДАНИЕ

**Часть 1. Анализ графиков обучения** (аналогично, но с стоматологическими примерами).

**Часть 2. Выбор метрики для стоматологических задач:** скрининг острого периодонтита, определение необходимости удаления, оценка риска осложнений после имплантации.

**Часть 3. Рефлексия о переобучении в клинической практике:** может ли стоматолог «переобучиться»? Как избежать?

**Часть 4. Этический анализ:** вам предлагают модель для диагностики кариеса по фотографиям. Метрики: accuracy 92%, precision 88%, recall 70%. Согласитесь ли использовать? Какие вопросы зададите?

---

## 11. ЛИТЕРАТУРА

(Оставить ту же, но можно добавить источники по стоматологии, если есть, но не обязательно)

1. Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep learning*. MIT Press.
2. Chollet, F. (2021). *Deep learning with Python* (2nd ed.). Manning Publications.
3. Géron, A. (2023). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly.
4. Scikit-learn Documentation (2024). *Model evaluation: metrics*.
5. Крылов, В. (2025). Метрики классификации в клинической медицине: выбор и интерпретация. *Журнал медицинской информатики*, 14(2), 45–53.
6. Hugging Face Documentation (2024). *Fine-tuning with Trainer*.
7. Севостьянов, А. (2024). Интерпретация графиков обучения нейросетей: руководство для врачей. *Цифровая медицина*, 9(1), 78–89.
8. Devlin, J. et al. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding.
9. Vaswani, A. et al. (2017). Attention is all you need.
10. World Health Organization. (2023). *Oral health*. WHO. (добавим для стоматологии)
